### Game Dashboard Setup

Creates an AI/BI Dashboard for Level 2 of Casper's Kitchen Rescue:
- **Level 2**: Delivery timing dashboard (The Slow Kitchen)

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

##### Create dashboard via Lakeview API

We create the dashboard programmatically with pre-built queries that
surface the delivery timing data. Players need to interact with the
charts and filters to find the anomaly.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import Dashboard
import json

w = WorkspaceClient()

# Get warehouse
warehouses = list(w.warehouses.list())
warehouse_id = None
for wh in warehouses:
    if wh.name and CATALOG in wh.name:
        warehouse_id = wh.id
        break
if not warehouse_id:
    for wh in warehouses:
        if wh.enable_serverless_compute or 'serverless' in (wh.name or '').lower():
            warehouse_id = wh.id
            break
if not warehouse_id and warehouses:
    warehouse_id = warehouses[0].id

print(f"Using warehouse_id={warehouse_id}")

In [ ]:
# Define the Lakeview dashboard serialized definition
# Uses PRE-AGGREGATED datasets (skill: bar/line charts need disaggregated: true with pre-aggregated data)
# Raw aggregation in widgets (disaggregated: false) often shows "no data" - pre-aggregate in SQL instead

dashboard_definition = {
    "datasets": [
        {
            "name": "by_location",
            "displayName": "By Location",
            "queryLines": [
                f"SELECT location_name, ROUND(AVG(kitchen_prep_minutes), 1) AS avg_kitchen_prep, ",
                f"ROUND(AVG(total_delivery_minutes), 1) AS avg_total_delivery ",
                f"FROM {CATALOG}.game.investigation_delivery_times GROUP BY location_name"
            ]
        },
        {
            "name": "by_day_location",
            "displayName": "By Day and Location",
            "queryLines": [
                f"SELECT order_day, location_name, ROUND(AVG(kitchen_prep_minutes), 1) AS avg_prep, ",
                f"ROUND(AVG(total_delivery_minutes), 1) AS avg_delivery ",
                f"FROM {CATALOG}.game.investigation_delivery_times GROUP BY order_day, location_name"
            ]
        }
    ],
    "pages": [
        {
            "name": "investigation",
            "displayName": "Kitchen Investigation",
            "pageType": "PAGE_TYPE_CANVAS",
            "layout": [
                {
                    "widget": {
                        "name": "narrative",
                        "multilineTextboxSpec": {
                            "lines": [
                                "## 🔍 The Slow Kitchen Investigation\n\nCustomers are complaining about cold food. One kitchen is significantly slower than others. **Use the charts below to identify:**\n1. Which location has abnormally long kitchen prep times?\n2. When did the slowdown start?\n\n*Tip: Compare the kitchen prep time trends across locations. Look for a divergence.*"
                            ]
                        }
                    },
                    "position": {"x": 0, "y": 0, "width": 6, "height": 2}
                },
                {
                    "widget": {
                        "name": "avg_prep_by_location",
                        "queries": [
                            {
                                "name": "main_query",
                                "query": {
                                    "datasetName": "by_location",
                                    "disaggregated": True,
                                    "fields": [
                                        {"name": "location_name", "expression": "`location_name`"},
                                        {"name": "avg_kitchen_prep", "expression": "`avg_kitchen_prep`"}
                                    ]
                                }
                            }
                        ],
                        "spec": {
                            "version": 3,
                            "widgetType": "bar",
                            "encodings": {
                                "x": {"fieldName": "location_name", "scale": {"type": "categorical"}, "displayName": "Location"},
                                "y": {"fieldName": "avg_kitchen_prep", "scale": {"type": "quantitative"}, "displayName": "Avg Kitchen Prep (min)"}
                            },
                            "frame": {"showTitle": True, "title": "Avg Kitchen Prep by Location"}
                        }
                    },
                    "position": {"x": 0, "y": 2, "width": 3, "height": 3}
                },
                {
                    "widget": {
                        "name": "avg_delivery_by_location",
                        "queries": [
                            {
                                "name": "main_query",
                                "query": {
                                    "datasetName": "by_location",
                                    "disaggregated": True,
                                    "fields": [
                                        {"name": "location_name", "expression": "`location_name`"},
                                        {"name": "avg_total_delivery", "expression": "`avg_total_delivery`"}
                                    ]
                                }
                            }
                        ],
                        "spec": {
                            "version": 3,
                            "widgetType": "bar",
                            "encodings": {
                                "x": {"fieldName": "location_name", "scale": {"type": "categorical"}, "displayName": "Location"},
                                "y": {"fieldName": "avg_total_delivery", "scale": {"type": "quantitative"}, "displayName": "Avg Total Delivery (min)"}
                            },
                            "frame": {"showTitle": True, "title": "Avg Total Delivery by Location"}
                        }
                    },
                    "position": {"x": 3, "y": 2, "width": 3, "height": 3}
                },
                {
                    "widget": {
                        "name": "prep_trend_by_day",
                        "queries": [
                            {
                                "name": "main_query",
                                "query": {
                                    "datasetName": "by_day_location",
                                    "disaggregated": True,
                                    "fields": [
                                        {"name": "order_day", "expression": "`order_day`"},
                                        {"name": "location_name", "expression": "`location_name`"},
                                        {"name": "avg_prep", "expression": "`avg_prep`"}
                                    ]
                                }
                            }
                        ],
                        "spec": {
                            "version": 3,
                            "widgetType": "line",
                            "encodings": {
                                "x": {"fieldName": "order_day", "scale": {"type": "temporal"}, "displayName": "Date"},
                                "y": {"fieldName": "avg_prep", "scale": {"type": "quantitative"}, "displayName": "Avg Kitchen Prep (min)"},
                                "color": {"fieldName": "location_name", "scale": {"type": "categorical"}, "displayName": "Location"}
                            },
                            "frame": {"showTitle": True, "title": "Kitchen Prep Trend by Day"}
                        }
                    },
                    "position": {"x": 0, "y": 5, "width": 6, "height": 3}
                },
                {
                    "widget": {
                        "name": "delivery_trend_by_day",
                        "queries": [
                            {
                                "name": "main_query",
                                "query": {
                                    "datasetName": "by_day_location",
                                    "disaggregated": True,
                                    "fields": [
                                        {"name": "order_day", "expression": "`order_day`"},
                                        {"name": "location_name", "expression": "`location_name`"},
                                        {"name": "avg_delivery", "expression": "`avg_delivery`"}
                                    ]
                                }
                            }
                        ],
                        "spec": {
                            "version": 3,
                            "widgetType": "line",
                            "encodings": {
                                "x": {"fieldName": "order_day", "scale": {"type": "temporal"}, "displayName": "Date"},
                                "y": {"fieldName": "avg_delivery", "scale": {"type": "quantitative"}, "displayName": "Avg Total Delivery (min)"},
                                "color": {"fieldName": "location_name", "scale": {"type": "categorical"}, "displayName": "Location"}
                            },
                            "frame": {"showTitle": True, "title": "Total Delivery Trend by Day"}
                        }
                    },
                    "position": {"x": 0, "y": 8, "width": 6, "height": 3}
                }
            ]
        }
    ]
}

print("Dashboard definition ready (pre-aggregated datasets)")
print(f"  Datasets: {[d['name'] for d in dashboard_definition['datasets']]}")
print(f"  Widgets: {[w['widget']['name'] for w in dashboard_definition['pages'][0]['layout']]}")

In [ ]:
host = w.config.host.rstrip("/")
dash_title = f"Casper's Kitchen Investigation - The Slow Kitchen ({CATALOG})"
final_url = None
dashboard_id = None

# 1. Check for existing dashboard
print("Checking for existing dashboard...")
try:
    existing = list(w.lakeview.list())
    for d in existing:
        if d.display_name == dash_title:
            dashboard_id = d.dashboard_id
            print(f"♻️ Reusing existing dashboard (id={dashboard_id})")
            print("  (Run cleanup first if you need the latest definition with scale encodings)")
            break
except Exception as e:
    print(f"Could not list dashboards: {e}")

# 2. Create via SDK only if no existing dashboard (Lakeview update API does not support serialized_dashboard)
if not dashboard_id:
    print("Creating dashboard via SDK...")
    try:
        dashboard = w.lakeview.create(
            Dashboard(
                display_name=dash_title,
                warehouse_id=warehouse_id,
                serialized_dashboard=json.dumps(dashboard_definition),
            )
        )
        dashboard_id = dashboard.dashboard_id
        print(f"\u2705 Created dashboard (id={dashboard_id})")
    except Exception as e:
        print(f"\u26a0\ufe0f SDK create failed: {e}")

# 3. Fallback: REST API
if not dashboard_id:
    print("Trying REST API fallback...")
    try:
        token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    except Exception:
        token = w.config.token
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    try:
        import requests
        resp = requests.post(
            f"{host}/api/2.0/lakeview/dashboards",
            headers=headers,
            json={
                "display_name": dash_title,
                "warehouse_id": warehouse_id,
                "serialized_dashboard": json.dumps(dashboard_definition),
            },
        )
        print(f"  Response status: {resp.status_code}")
        print(f"  Response body: {resp.text[:500]}")
        resp.raise_for_status()
        dashboard_id = resp.json().get("dashboard_id")
        if dashboard_id:
            print(f"\u2705 Created dashboard via REST (id={dashboard_id})")
    except Exception as e:
        print(f"\u26a0\ufe0f REST API also failed: {e}")

# 4. Publish and build published URL (game should open read-only published view, not editable draft)
if dashboard_id:
    try:
        w.lakeview.publish(
            dashboard_id=dashboard_id,
            warehouse_id=warehouse_id,
            embed_credentials=True,
        )
        print("\u2705 Published dashboard")
    except Exception as pub_err:
        print(f"\u26a0\ufe0f Could not publish: {pub_err}")
    final_url = f"{host}/dashboardsv3/{dashboard_id}/published"

if not final_url:
    print("\n\u274c Could not create dashboard.")
    print(f"Create one manually, then run:")
    print(f"  MERGE INTO {CATALOG}.game.config AS t")
    print(f"  USING (SELECT 'dashboard_url' AS config_key, '<URL>' AS config_value) AS s")
    print(f"  ON t.config_key = s.config_key")
    print(f"  WHEN MATCHED THEN UPDATE SET config_value = s.config_value")
    print(f"  WHEN NOT MATCHED THEN INSERT VALUES (s.config_key, s.config_value);")


In [ ]:
# Store dashboard URL in game config and register with uc_state for cleanup
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.game.config (
  config_key STRING,
  config_value STRING
)
""")

if dashboard_id:
    import sys
    sys.path.append('../utils')
    from uc_state import add
    add(CATALOG, "dashboards", {"dashboard_id": dashboard_id, "display_name": dash_title})

if final_url:
    spark.sql(f"""
    MERGE INTO {CATALOG}.game.config AS target
    USING (SELECT 'dashboard_url' AS config_key, '{final_url}' AS config_value) AS source
    ON target.config_key = source.config_key
    WHEN MATCHED THEN UPDATE SET config_value = source.config_value
    WHEN NOT MATCHED THEN INSERT (config_key, config_value) VALUES (source.config_key, source.config_value)
    """)
    print(f"\u2705 Stored dashboard URL in game config")
else:
    print("\u26a0\ufe0f No dashboard URL to store. Set it manually in game.config.")

In [ ]:
print("\u2705 Dashboard setup complete")